# OrchestraLLM on Google Colab
**Paper:** *OrchestraLLM: Efficient Orchestration of Language Models for Dialogue State Tracking*

**Backbone:** FLAN-T5-large (SLM) + Claude Haiku (LLM) + SenBERT Router

---
### ⚠️ Before you start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. You need a free [Anthropic API key](https://console.anthropic.com/) for the LLM (IC-DST) component
3. Runtime resets clear `/content` — use **Google Drive mounting** (Cell 2) to persist checkpoints

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 2 — Mount Google Drive (recommended to persist checkpoints)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
# All persistent data will live here — survives runtime resets
DRIVE_DIR = '/content/drive/MyDrive/orchestrallm'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Working dir: {DRIVE_DIR}')

Mounted at /content/drive
Drive mounted. Working dir: /content/drive/MyDrive/orchestrallm


## Cell 3 — Install dependencies

In [2]:
%%capture
!pip install transformers>=4.40.0 sentence-transformers>=2.6.0 \
             datasets>=2.18.0 accelerate>=0.27.0 anthropic>=0.25.0 \
             pyyaml tqdm
print('Done installing.')

## Cell 4 — Upload project files
Upload your 7 Python files + config.yaml here (or pull from GitHub if you've pushed them).

In [6]:
# ── Option A: Upload files (Full or Partial) ──────────────────────────────────
# Modified to handle Colab's auto-renaming (e.g., 'file (1).py') and overwrite originals.

from google.colab import files
import os, shutil, re

PROJECT_DIR = '/content/orchestrallm'
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Select files to upload (existing files will be overwritten):")
uploaded = files.upload()

for fname in uploaded:
    # 1. Strip the Colab suffix ' (1)' if it exists to find the intended filename
    # This regex looks for ' (number)' before the extension
    target_name = re.sub(r'\s\(\d+\)(?=\.\w+$)|\s\(\d+\)$', '', fname)

    dest_path = os.path.join(PROJECT_DIR, target_name)
    src_path = os.path.join('/content', fname)

    # 2. Move and overwrite
    if os.path.exists(src_path):
        shutil.move(src_path, dest_path)
        print(f'Updated: {dest_path}')

os.chdir(PROJECT_DIR)
print('\nCurrent files in project directory:')
!ls -lh {PROJECT_DIR}

Select files to upload (existing files will be overwritten):


Saving orchestrallm.py to orchestrallm.py
Saving prompt_dst.py to prompt_dst.py
Saving README.md to README.md
Saving requirements.txt to requirements.txt

Current files in project directory:
total 116K
-rw-r--r-- 1 root root 3.8K Mar 14 07:05 config.yaml
-rw-r--r-- 1 root root  16K Mar 14 07:05 data_preprocessing.py
-rw-r--r-- 1 root root 9.3K Mar 14 07:05 evaluate.py
-rw-r--r-- 1 root root  14K Mar 14 07:05 ic_dst.py
-rw-r--r-- 1 root root  13K Mar 14 07:06 orchestrallm.py
-rw-r--r-- 1 root root  20K Mar 14 07:06 prompt_dst.py
-rw-r--r-- 1 root root 3.5K Mar 14 07:06 README.md
-rw-r--r-- 1 root root  236 Mar 14 07:06 requirements.txt
-rw-r--r-- 1 root root  23K Mar 14 07:04 router.py


In [ ]:
import os, re, shutil

PROJECT_DIR = '/content/orchestrallm'
os.chdir(PROJECT_DIR)

# Pattern to match files with Colab version suffixes like ' (1)'
pattern = re.compile(r'.*\s\(\d+\)\.py$')

print('Cleaning up project directory...')
for fname in os.listdir(PROJECT_DIR):
    if pattern.match(fname):
        os.remove(os.path.join(PROJECT_DIR, fname))
        print(f'Removed redundant file: {fname}')

print('\nCleaned project directory:')
!ls -lh {PROJECT_DIR}

Cleaning up project directory...

Cleaned project directory:
total 120K
-rw-r--r-- 1 root root 3.5K Mar  6 09:43 config.yaml
-rw-r--r-- 1 root root  16K Mar  6 09:43 data_preprocessing.py
-rw-r--r-- 1 root root 9.3K Mar  6 09:43 evaluate.py
-rw-r--r-- 1 root root  14K Mar  6 09:43 ic_dst.py
-rw-r--r-- 1 root root  13K Mar  6 09:43 orchestrallm.py
-rw-r--r-- 1 root root  19K Mar  6 09:43 prompt_dst.py
-rw-r--r-- 1 root root 6.4K Mar  6 09:43 README.md
-rw-r--r-- 1 root root  236 Mar  6 09:43 requirements.txt
-rw-r--r-- 1 root root  23K Mar  6 09:43 router.py


In [ ]:
# ── Option B: Pull from GitHub (if you've pushed the files) ──────────────────
# Uncomment and replace with your repo URL

# !git clone https://github.com/YOUR_USERNAME/orchestrallm.git /content/orchestrallm
# os.chdir('/content/orchestrallm')

## Cell 5 — Set API key and configure paths

In [7]:
import os
from getpass import getpass

# Set your Anthropic API key (only needed for IC-DST / LLM component)
os.environ['ANTHROPIC_API_KEY'] = getpass('Enter your Anthropic API key: ')
print('API key set.')

Enter your Anthropic API key: ··········
API key set.


In [8]:
import yaml

DRIVE_DIR = '/content/drive/MyDrive/orchestrallm'  # Persistent
PROJECT_DIR = '/content/orchestrallm'               # Code lives here

# Update config.yaml to point paths to Google Drive (persistent across resets)
with open(f'{PROJECT_DIR}/config.yaml') as f:
    config = yaml.safe_load(f)

config['paths'] = {
    'data_dir':       f'{DRIVE_DIR}/data/multiwoz',
    'processed_dir':  f'{DRIVE_DIR}/data/processed',
    'model_dir':      f'{DRIVE_DIR}/models/prompt_dst',
    'expert_pool_dir':f'{DRIVE_DIR}/data/expert_pools',
    'results_dir':    f'{DRIVE_DIR}/results',
}

with open(f'{PROJECT_DIR}/config.yaml', 'w') as f:
    yaml.dump(config, f)

# Create all directories
for path in config['paths'].values():
    os.makedirs(path, exist_ok=True)

print('Paths configured:')
for k, v in config['paths'].items():
    print(f'  {k}: {v}')

Paths configured:
  data_dir: /content/drive/MyDrive/orchestrallm/data/multiwoz
  processed_dir: /content/drive/MyDrive/orchestrallm/data/processed
  model_dir: /content/drive/MyDrive/orchestrallm/models/prompt_dst
  expert_pool_dir: /content/drive/MyDrive/orchestrallm/data/expert_pools
  results_dir: /content/drive/MyDrive/orchestrallm/results


## Cell 6 — Download MultiWOZ 2.4
This downloads the dataset directly from the official source (~50MB).

In [9]:
import os, zipfile

DATA_DIR = config['paths']['data_dir']
os.makedirs(DATA_DIR, exist_ok=True)
data_file = f'{DATA_DIR}/data.json'

if os.path.exists(data_file):
    print(f'data.json already exists ({os.path.getsize(data_file)/1e6:.1f} MB) — skipping download.')
else:
    print('Downloading MultiWOZ 2.4...')
    # Download from the MultiWOZ 2.4 GitHub release
    !wget -q --show-progress -O /tmp/multiwoz24.zip \
        https://github.com/smartyfh/MultiWOZ2.4/raw/main/data/MULTIWOZ2.4.zip

    print('Extracting...')
    with zipfile.ZipFile('/tmp/multiwoz24.zip', 'r') as z:
        z.extractall('/tmp/multiwoz24/')

    # Find and move data.json
    import glob, shutil
    candidates = glob.glob('/tmp/multiwoz24/**/data.json', recursive=True)
    if candidates:
        shutil.copy(candidates[0], data_file)
        print(f'data.json saved → {data_file}')
    else:
        print('ERROR: data.json not found in zip. Files found:')
        !find /tmp/multiwoz24 -name '*.json' | head -20

    # Extract official split files (required for train/val/test splitting)
    for split_file in ['valListFile.json', 'testListFile.json']:
        matches = glob.glob(f'/tmp/multiwoz24/**/{split_file}', recursive=True)
        if matches:
            shutil.copy(matches[0], f'{DATA_DIR}/{split_file}')
            print(f'{split_file} saved → {DATA_DIR}/{split_file}')
        else:
            print(f'WARNING: {split_file} not found in zip')

import json
with open(data_file) as f:
    d = json.load(f)
print(f'Loaded {len(d)} dialogues from MultiWOZ 2.4')

data.json already exists (276.6 MB) — skipping download.
Loaded 10438 dialogues from MultiWOZ 2.4


## Cell 7 — Preprocess Data

In [10]:
os.chdir(PROJECT_DIR)

processed_dir = config['paths']['processed_dir']
train_file = f'{processed_dir}/train.jsonl'

if os.path.exists(train_file):
    import subprocess
    n = int(subprocess.check_output(['wc', '-l', train_file]).split()[0])
    print(f'Preprocessed data already exists ({n} train turns) — skipping.')
else:
    print('Preprocessing MultiWOZ...')
    !python data_preprocessing.py \
        --data_dir  {config['paths']['data_dir']} \
        --out_dir   {config['paths']['processed_dir']} \
        --config    config.yaml

# Show split sizes
for split in ['train', 'holdout', 'val', 'test']:
    path = f"{config['paths']['processed_dir']}/{split}.jsonl"
    if os.path.exists(path):
        !echo -n "{split}: " && wc -l < {path} | xargs echo "turns"

Preprocessing MultiWOZ...
Loading from /content/drive/MyDrive/orchestrallm/data/multiwoz/data.json
Loaded 10438 dialogues from /content/drive/MyDrive/orchestrallm/data/multiwoz
Split — train: 521, holdout: 100, val: 9817

[train]
  Dialogues : 521
  Turns     : 3595
  Turns with non-empty TLB : 2348 (65.3%)
  Top-5 slots: [('restaurant-semi-food', 248), ('train-semi-departure', 197), ('train-semi-destination', 192), ('train-semi-day', 189), ('restaurant-semi-area', 185)]
Saved 3595 examples → /content/drive/MyDrive/orchestrallm/data/processed/train.jsonl

[holdout]
  Dialogues : 100
  Turns     : 661
  Turns with non-empty TLB : 421 (63.7%)
  Top-5 slots: [('restaurant-semi-food', 47), ('restaurant-semi-area', 41), ('restaurant-semi-pricerange', 38), ('train-semi-destination', 34), ('train-semi-departure', 33)]
Saved 661 examples → /content/drive/MyDrive/orchestrallm/data/processed/holdout.jsonl

[val]
  Dialogues : 9817
  Turns     : 67268
  Turns with non-empty TLB : 43674 (64.9%)
  

## Cell 8 — Fine-tune Prompt-DST (FLAN-T5-large)
⏱️ **Estimated time:** ~2-4 hours on T4 GPU (5% few-shot data, 10 epochs)

💡 **Tip:** Colab Pro disconnects after ~12h. The best checkpoint is saved to Drive after each epoch — you can resume from there.

In [11]:
model_dir = config['paths']['model_dir']
best_ckpt = f'{model_dir}/best'

if os.path.exists(f'{best_ckpt}/config.json'):
    print(f'Checkpoint already exists at {best_ckpt} — skipping training.')
    print('Delete it and rerun this cell to retrain.')
else:
    print('Starting FLAN-T5-large fine-tuning...')
    print('Checkpoints saved to Drive after each epoch.')
    !python prompt_dst.py train --config config.yaml

Starting FLAN-T5-large fine-tuning...
Checkpoints saved to Drive after each epoch.
[PromptDST] Using device: cuda
[PromptDST] Backbone: google/flan-t5-large
[PromptDST] Loading from: google/flan-t5-large
config.json: 100% 662/662 [00:00<00:00, 2.75MB/s]
tokenizer_config.json: 2.54kB [00:00, 6.59MB/s]
spiece.model: 100% 792k/792k [00:01<00:00, 713kB/s] 
tokenizer.json: 2.42MB [00:00, 78.2MB/s]
special_tokens_map.json: 2.20kB [00:00, 6.14MB/s]
model.safetensors: 100% 3.13G/3.13G [00:21<00:00, 148MB/s]
Loading weights: 100% 558/558 [00:04<00:00, 115.20it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
generation_config.json: 100% 147/147 [00:00<00:00, 546kB/s]

[PromptDST] Starting training — 3595 train, 67268 val turns
[PromptDST] No existing checkp

## Cell 9 — Evaluate Prompt-DST (SLM only)

In [ ]:
!python prompt_dst.py eval \
    --config config.yaml \
    --checkpoint {config['paths']['model_dir']}/best

## Cell 10 — Generate holdout predictions for both experts
This runs both SLM and LLM on the holdout set and saves predictions — needed to build expert pools.

In [ ]:
import sys, json
sys.path.insert(0, PROJECT_DIR)

from data_preprocessing import load_jsonl, string_to_state, format_input, state_to_string

pool_dir = config['paths']['expert_pool_dir']
os.makedirs(pool_dir, exist_ok=True)
processed_dir = config['paths']['processed_dir']

holdout_ex = load_jsonl(f'{processed_dir}/holdout.jsonl')
print(f'Holdout set: {len(holdout_ex)} turns')

In [ ]:
# ── SLM predictions on holdout ────────────────────────────────────────────────
slm_preds_path = f'{pool_dir}/slm_holdout_preds.json'

if os.path.exists(slm_preds_path):
    print('SLM holdout predictions already exist — skipping.')
else:
    import yaml, torch
    from prompt_dst import PromptDST
    from collections import defaultdict
    from tqdm.notebook import tqdm

    with open(f'{PROJECT_DIR}/config.yaml') as f:
        cfg = yaml.safe_load(f)

    slm = PromptDST(cfg)
    slm.load(f"{cfg['paths']['model_dir']}/best")

    # Group by dialogue to track accumulated DST correctly
    dialogues = defaultdict(list)
    for ex in holdout_ex:
        dialogues[ex['dialogue_id']].append(ex)
    for did in dialogues:
        dialogues[did].sort(key=lambda x: x['turn_idx'])

    slm_preds = {}
    for did, turns in tqdm(dialogues.items(), desc='SLM holdout'):
        acc_dst = {}
        for turn in turns:
            key = f"{turn['dialogue_id']}_{turn['turn_idx']}"
            inp = format_input(acc_dst, turn['agent_utt'], turn['user_utt'])
            pred_str = slm.predict_turn(inp)
            slm_preds[key] = pred_str
            acc_dst.update(string_to_state(pred_str))

    with open(slm_preds_path, 'w') as f:
        json.dump(slm_preds, f)
    print(f'SLM predictions saved ({len(slm_preds)} turns)')

In [ ]:
# ── LLM predictions on holdout ────────────────────────────────────────────────
# ⚠️  This makes ~N_holdout_turns API calls to Claude. Costs ~$0.50-2 depending on size.

llm_preds_path = f'{pool_dir}/llm_holdout_preds.json'

if os.path.exists(llm_preds_path):
    print('LLM holdout predictions already exist — skipping.')
else:
    import yaml
    from ic_dst import ICDST
    from collections import defaultdict
    from tqdm.notebook import tqdm

    with open(f'{PROJECT_DIR}/config.yaml') as f:
        cfg = yaml.safe_load(f)

    train_ex = load_jsonl(f"{cfg['paths']['processed_dir']}/train.jsonl")
    llm = ICDST(cfg)
    llm.load_exemplar_pool(train_ex)

    dialogues = defaultdict(list)
    for ex in holdout_ex:
        dialogues[ex['dialogue_id']].append(ex)
    for did in dialogues:
        dialogues[did].sort(key=lambda x: x['turn_idx'])

    llm_preds = {}
    for did, turns in tqdm(list(dialogues.items()), desc='LLM holdout'):
        acc_dst = {}
        for turn in turns:
            key = f"{turn['dialogue_id']}_{turn['turn_idx']}"
            _, pred_str = llm.predict_turn(acc_dst, turn['agent_utt'], turn['user_utt'], did)
            llm_preds[key] = pred_str
            acc_dst.update(string_to_state(pred_str))

    with open(llm_preds_path, 'w') as f:
        json.dump(llm_preds, f)
    print(f'LLM predictions saved ({len(llm_preds)} turns)')

## Cell 11 — Build Expert Pools

In [ ]:
!python router.py build_pools --config config.yaml

## Cell 12 — Fine-tune SenBERT Retriever (optional)
Adds ~3% TLB JGA improvement per paper results. Takes ~15-30 min on T4.

In [ ]:
retriever_path = f"{config['paths']['expert_pool_dir']}/retriever"

if os.path.exists(retriever_path):
    print('Retriever already fine-tuned — skipping.')
else:
    !python router.py train_retriever --config config.yaml

## Cell 13 — Full OrchestraLLM Evaluation

In [ ]:
results_dir = config['paths']['results_dir']
os.makedirs(results_dir, exist_ok=True)

# Limit to 30 dialogues to control LLM API cost (~$0.20)
# Remove --max_dialogues to run on full val set
!python orchestrallm.py eval \
    --config config.yaml \
    --max_dialogues 30 \
    --results_out {results_dir}/orchestrallm_results.json

In [ ]:
# Pretty-print results
import json
with open(f"{config['paths']['results_dir']}/orchestrallm_results.json") as f:
    results = json.load(f)

print('\n' + '='*50)
print('  OrchestraLLM Results')
print('='*50)
print(f"  TLB JGA          : {results['tlb_jga']:.2f}%")
print(f"  DST JGA          : {results['dst_jga']:.2f}%")
print(f"  SLM assignment   : {results['slm_assignment_ratio']:.1f}%  (cheaper model)")
print(f"  LLM assignment   : {results['llm_assignment_ratio']:.1f}%")
print(f"  N turns          : {results['n_turns']}")
print('='*50)

## Cell 14 — Interactive Single-Turn Demo

In [ ]:
import sys, yaml
sys.path.insert(0, PROJECT_DIR)

from orchestrallm import OrchestraLLM
from data_preprocessing import load_jsonl, state_to_string

with open(f'{PROJECT_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)

train_ex = load_jsonl(f"{cfg['paths']['processed_dir']}/train.jsonl")

pipeline = OrchestraLLM(cfg)
pipeline.load(exemplar_examples=train_ex)
print('Pipeline ready!')

In [ ]:
# ── Run a sample from the MultiWOZ example in the paper ──────────────────────
from data_preprocessing import string_to_state

# Simulating SNG01856.json turns
test_turns = [
    {
        'agent_utt': '',
        'user_utt':  'I am looking for a place to stay that has cheap price range, it should be a hotel.',
        'prev_dst':  {},
        'target':    'hotel-semi-type = hotel | hotel-semi-pricerange = cheap',
    },
    {
        'agent_utt': 'Okay, do you have a specific area you want to stay in?',
        'user_utt':  'No, I just need to make sure it is cheap. Oh, and I need parking.',
        'prev_dst':  {'hotel-semi-type': 'hotel', 'hotel-semi-pricerange': 'cheap'},
        'target':    'hotel-semi-parking = yes',
    },
    {
        'agent_utt': 'I found 1 cheap hotel for you that includes parking. Shall I book it?',
        'user_utt':  'Yes please. 6 people, 3 nights starting on Tuesday.',
        'prev_dst':  {'hotel-semi-type': 'hotel', 'hotel-semi-pricerange': 'cheap', 'hotel-semi-parking': 'yes'},
        'target':    'hotel-book-people = 6 | hotel-book-stay = 3 | hotel-book-day = tuesday',
    },
]

print('Running OrchestraLLM on example dialogue from paper (SNG01856)...\n')
accumulated_dst = {}

for i, turn in enumerate(test_turns):
    expert, pred_tlb, pred_str = pipeline.predict_turn(
        prev_dst=accumulated_dst,
        agent_utt=turn['agent_utt'],
        user_utt=turn['user_utt'],
    )
    accumulated_dst.update(pred_tlb)

    print(f'── Turn {i+1} ──')
    print(f'  User    : {turn["user_utt"]}')
    print(f'  Expert  : {expert.upper()}')
    print(f'  Predicted TLB : {pred_str or "[none]"}')
    print(f'  Gold TLB      : {turn["target"]}')
    print(f'  Full DST      : {state_to_string(accumulated_dst)}')
    print()

## Cell 15 — Download results from Colab
If not using Drive, download results to your local machine.

In [ ]:
from google.colab import files
import os

results_file = f"{config['paths']['results_dir']}/orchestrallm_results.json"
if os.path.exists(results_file):
    files.download(results_file)
else:
    print('No results file found. Run Cell 13 first.')

---
## Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA out of memory` | Reduce `batch_size` in config.yaml (try 4) or `gradient_accumulation_steps: 8` |
| `Runtime disconnected` | Checkpoints are on Drive — just reconnect and re-run from Cell 8 (it will skip if checkpoint exists) |
| `anthropic.AuthenticationError` | Re-run Cell 5 to re-enter your API key (it doesn't persist across sessions) |
| `data.json not found` | MultiWOZ zip structure may have changed — manually upload `data.json` to Drive path |
| Slow training | Normal for T4 — consider Colab Pro for A100 (~4x faster) |